# Governance decision trace

One decision, end to end, through the trust boundary. Replaces `scripts/demo_end_to_end.py`.

The carried-forward case: the catalog asserts `certified: true` while the governing policy is violated. Two internally consistent systems of record; nobody lying; the organisation wrong in a way neither can see alone.

## Setup

Redacted readiness report — `config.py` never prints a secret.

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")          # Neo4j emits schema hints we handle
import config
print(config.report())

readiness
  neo4j      uri=neo4j+s://<redacted>.databases.neo4j.io
             user fp=6b611cde  db fp=6b611cde
             password fp=58ebef03
             configured=True
  qdrant     url=https://<redacted>.sa-east-1-0.aws.cloud.qdrant.io
             api_key fp=f3da5a0f
             configured=True
  embedding  provider=bedrock  model=amazon.titan-embed-text-v2:0  dim=1024
             configured=True
  trust      policy=v1.0  floor=0.75  min_tier=ingested


## Facts across three tiers

One is deliberately ineligible.

In [2]:
from datetime import datetime, timedelta, timezone
from biomedical.foundation import (Fact, ResolutionMethod, ReviewState, Tier, Timings,
                                   TrustPolicy, ValidityInterval, Verdict, RuleOutcome,
                                   RuleResult, Severity, asserted, assemble, derived, ingested)

NOW = datetime.now(timezone.utc)
cfg = config.load()
policy = TrustPolicy(version=cfg.trust.policy_version,
                     confidence_floor=cfg.trust.confidence_floor)

facts = [
    Fact(key="dq_rule_count", value="0", entity_id="table:CustomerMaster",
         stamp=ingested("catalog", "2026-10", NOW),
         validity=ValidityInterval(valid_from=NOW - timedelta(days=4)), recorded_at=NOW),
    Fact(key="certified", value="true", entity_id="table:CustomerMaster",
         stamp=ingested("catalog", "2026-10", NOW),
         validity=ValidityInterval(valid_from=NOW - timedelta(days=4)), recorded_at=NOW),
    Fact(key="consent_status", value="Withdrawn", entity_id="patient:PAT015",
         stamp=asserted("authored", "synthetic-v1", NOW),
         validity=ValidityInterval(valid_from=NOW - timedelta(days=200)), recorded_at=NOW),
    Fact(key="rxcui", value="1657993", entity_id="patient:PAT015",
         stamp=derived("resolver", "1.0", NOW, ResolutionMethod.MULTI_CANDIDATE,
                       "'pembrolizumab 100mg' matched 3 RxNorm candidates"),
         validity=ValidityInterval(valid_from=NOW - timedelta(days=1)), recorded_at=NOW,
         review_state=ReviewState.NEEDS_REVIEW),
]
for f in facts:
    print(" ", f.describe())

  dq_rule_count='0' | ingested · catalog@2026-10 | 2026-10-07 -> open
  certified='true' | ingested · catalog@2026-10 | 2026-10-07 -> open
  consent_status='Withdrawn' | asserted · authored@synthetic-v1 | 2026-03-25 -> open
  rxcui='1657993' | derived · resolver@1.0 · conf=0.30 · method=multiple_candidates | 2026-10-10 -> open | NEEDS_REVIEW


## Through the trust boundary

In [3]:
bundle = assemble(facts, policy, NOW)
print(bundle.describe())
print("\neligible:")
for k in bundle.facts:
    print("  ", k)
print("\nwithheld — the audit trail v1 had no equivalent of:")
for w in bundle.withheld:
    print("  ", w.describe())

FactBundle(3 eligible, 1 withheld, hash=d44bbd591ced, policy=v1.0)

eligible:
   dq_rule_count
   certified
   consent_status

withheld — the audit trail v1 had no equivalent of:
   rxcui withheld: tier_too_weak, confidence_below_floor, awaiting_review


## Evaluate

Rules receive a bundle and hold no store client, so there is no code path to an unfiltered query.

In [4]:
results = []

found, missing = bundle.require("dq_rule_count")
results.append(RuleResult("governance.hub_table_without_dq_rule",
    RuleOutcome.VIOLATION if found and found[0].value == "0" else RuleOutcome.CANNOT_EVALUATE,
    Severity.BLOCK, message="zero DQ rules for a hub-layer table",
    policy_iri="graph:Policy/POL_HUB_DQ", missing_dependencies=tuple(missing)))

found, missing = bundle.require("consent_status")
results.append(RuleResult("consent.withdrawn_patient_data",
    RuleOutcome.VIOLATION if found and str(found[0].value) == "Withdrawn" else RuleOutcome.CANNOT_EVALUATE,
    Severity.BLOCK, message="consent withdrawn", policy_iri="bio:POL001",
    missing_dependencies=tuple(missing)))

_, missing = bundle.require("rxcui")
results.append(RuleResult("safety.drug_without_external_identifier",
    RuleOutcome.CANNOT_EVALUATE, Severity.WARN,
    message="identifier unavailable at the required confidence",
    missing_dependencies=tuple(missing)))

verdict = Verdict(
    question="Certify Customer Master so I can publish PAT015's response in the Q3 report.",
    results=tuple(results), facts_relied_on=bundle.references(),
    facts_withheld=bundle.withheld, trust_policy_version=bundle.trust_policy_version,
    source_versions=bundle.source_versions(),
    timings=Timings.of(facts_ms=389.41, rules_ms=0.05), decided_at=NOW,
    bundle_hash=bundle.content_hash)

print(verdict.describe())

Q: Certify Customer Master so I can publish PAT015's response in the Q3 report.
decision: BLOCKED   policy v1.0
timing: 389.46 ms end-to-end (facts 389.41 ms · rules 0.05 ms — rule evaluation over already-materialised facts; excludes the graph round-trip)
rules:
  [X] governance.hub_table_without_dq_rule             violation (block)
  [X] consent.withdrawn_patient_data                   violation (block)
  [?] safety.drug_without_external_identifier          cannot_evaluate (warn)
        withheld dependency: rxcui
facts relied on:
  ingested  dq_rule_count='0' conf=1.00 catalog@2026-10 [2026-10-07 -> open]
  ingested  certified='true' conf=1.00 catalog@2026-10 [2026-10-07 -> open]
  asserted  consent_status='Withdrawn' conf=1.00 authored@synthetic-v1 [2026-03-25 -> open]
facts withheld:
  rxcui withheld: tier_too_weak, confidence_below_floor, awaiting_review


## The latency correction

v1 published **0.09 ms** as the cost of a verdict. A reviewer asked whether that included database access. It did not — roughly four orders of magnitude out. The component figure is now un-publishable without its qualifying note.

In [5]:
t = Timings.of(facts_ms=389.41, rules_ms=0.05)
print("headline:", t.headline())
print("full:    ", t.describe())
print()
print("decision:", verdict.decision(), "  inconclusive:", verdict.inconclusive)
print("-> CANNOT_EVALUATE means the action does not proceed on the rule's silence")

headline: 389.46 ms end-to-end
full:     389.46 ms end-to-end (facts 389.41 ms · rules 0.05 ms — rule evaluation over already-materialised facts; excludes the graph round-trip)

decision: BLOCKED   inconclusive: True
-> CANNOT_EVALUATE means the action does not proceed on the rule's silence
